In [ ]:
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from tools.embs_tools import get_embs, aggregate_embeddings, accelerated_cosine_similarity, get_crops_for_id, bb_weighted_average

In [ ]:
all_embs = get_embs("/home/simon/new_outputs/")
print("\n".join(list(all_embs.keys())))

In [ ]:
path_a = list(all_embs.keys())[0]
path_b = list(all_embs.keys())[-1]

print("------ Before aggregation ------")
print(f"Number of features in first file: {len(all_embs[path_a])}")
print(f"Number of features in second file: {len(all_embs[path_b])}")
all_embs_aggregated = {
    path: aggregate_embeddings(
        file_embs,
        aggregation_fn=bb_weighted_average
    )
    for path, file_embs in all_embs.items()
}
print("------ After aggregation ------")
print(f"Number of features in first file: {len(all_embs[path_a])}")
print(f"Number of features in second file: {len(all_embs[path_b])}")
print("------ Paths ------")
print(f"Path of first video: {path_a}")
print(f"Path of second video: {path_b}")

In [ ]:
embs_a = torch.stack([torch.tensor(emb) for _, emb in all_embs_aggregated[path_a]])
embs_b = torch.stack([torch.tensor(emb) for _, emb in all_embs_aggregated[path_b]])
ids_a = torch.tensor([id for id, _ in all_embs_aggregated[path_a]])
ids_b = torch.tensor([id for id, _ in all_embs_aggregated[path_b]])

similarity = accelerated_cosine_similarity(embs_a, embs_b, batch_size=2048)

In [ ]:
print(similarity.shape)
print(len(embs_a))
print(len(embs_b))

In [ ]:
maximum_index = torch.argsort(similarity.flatten(), descending=True)[10]
file_a_max_sim = int(maximum_index // similarity.shape[1])
file_b_max_sim = int(maximum_index % similarity.shape[1])

print("Max similarity indexes: ", (file_a_max_sim, file_b_max_sim))

In [ ]:
track_a_crops = get_crops_for_id(path_a, ids_a[file_a_max_sim])
track_b_crops = get_crops_for_id(path_b, ids_b[file_b_max_sim])

print("Len A: ", len(track_a_crops), "; Len B: ", len(track_b_crops))
print("Shape A: ", track_a_crops[0].shape if len(track_a_crops) > 0 else "No crops found")

In [ ]:
print("Path A: ", path_a)
print("Path B: ", path_b)
plt.imshow(cv2.cvtColor(track_a_crops[0], cv2.COLOR_BGR2RGB))
plt.show()
plt.imshow(cv2.cvtColor(track_b_crops[0], cv2.COLOR_BGR2RGB))
plt.show()